In [1]:
from prompt import vision_prompt
import os

In [2]:
os.chdir('/root/projects/purchases/purchases/')

In [7]:
from config.settings import LLM_API_KEY
from src.llm.parser import Receipt,clean_strip
from src.llm.vision import analyze_images_mcp

ImportError: cannot import name 'clean_strip' from 'src.llm.parser' (/root/projects/purchases/purchases/src/llm/parser.py)

In [4]:
from openai import OpenAI, AsyncOpenAI
minimax = OpenAI(base_url='https://api.minimax.io/v1',api_key=LLM_API_KEY)

In [5]:
request = "Can Aliens suffer? in one sentence"

response = minimax.chat.completions.create(
    model="MiniMax-M2.7",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": request},
    ],
    # Set reasoning_split=True to separate thinking content into reasoning_details field
    extra_body={"reasoning_split": True},
)

print(f"Thinking:\n{response.choices[0].message.reasoning_details[0]['text']}\n")
print(f"Text:\n{response.choices[0].message.content}\n")

Thinking:
The user is asking whether aliens can suffer, and wants the answer in one sentence. This is a philosophical question that touches on consciousness, the nature of suffering, and what we know (or don't know) about extraterrestrial life.

I should give a concise, thoughtful answer in one sentence.

Text:
If aliens possess consciousness and subjective experiences similar to humans, they could theoretically suffer, but since we have no evidence of extraterrestrial life or their neurological/conscious nature, this remains entirely speculative.



# Vision Model

In [6]:
from glob import glob

# --- Varianten für Bildpfade auf Windows ---

# A) Manuelle Liste mit raw-strings (r"...") oder normalen Slashes
bilder = [
    r"src/cloud/data/receipt_20260425_063827.jpg",  # Python akzeptiert auch /
]


# --- Analyse starten ---
ergebnisse = analyze_images_mcp(
    image_paths=bilder,
    api_key=LLM_API_KEY,
    prompt=vision_prompt
)



In [31]:
def clean_strip(raw_string):
    clean_string = raw_string.strip()
    if clean_string.startswith("```json"):
        clean_string = clean_string[7:] # Schneidet die ersten 7 Zeichen ab
    if clean_string.endswith("```"):
        clean_string = clean_string[:-3] # Schneidet die letzten 3 Zeichen ab
    
    clean_string = clean_string.strip()
    return clean_string

In [ ]:
ergebnisse[

# pydantiv parser

In [35]:
from pydantic import BaseModel
from typing import Optional, List, Any

# ============================================================
# Pydantic-Modelle für ALDI-Quittung (Pydantic v2)
# ============================================================

class Merchant(BaseModel):
    name: str
    address: str
    tax_id: Optional[str] = None
    phone: Optional[str] = None


class Transaction(BaseModel):
    date: str          # oder: from datetime import date → date
    time: str          # oder: from datetime import time → time
    currency: str
    total_amount: float
    payment_method: str
    card_last_four: Optional[str] = None
    receipt_number: Optional[str] = None


class Item(BaseModel):
    line_number: int
    sku: str
    name: str
    quantity: float
    unit: Optional[str] = None
    unit_price: float
    total_price: float
    tax_category: str
    discount: Optional[float] = None


class Metadata(BaseModel):
    ocr_confidence: str
    image_filename: Optional[str] = None


class Receipt(BaseModel):
    merchant: Merchant
    transaction: Transaction
    items: List[Item]
    taxes: List[Any] = []   # leer in deinem Beispiel; bei Bedarf eigenes Tax-Modell
    metadata: Metadata


# ============================================================
# Nutzung
# ============================================================

# Validieren & parsen
receipt = Receipt.model_validate(json.loads(clean_strip(ergebnisse[0]['text'])))

# Zugriff auf Felder
print(receipt.merchant.name)              # ALDI SÜD
print(receipt.transaction.total_amount)   # 63.59
print(len(receipt.items))                 # 41

# Zu JSON zurückwandeln (optional)
json_output = receipt.model_dump_json(indent=2)
print(json_output)

ALDI SÜD
63.59
41
{
  "merchant": {
    "name": "ALDI SÜD",
    "address": "Frankenforster Str. 21-25, 51427 Bergisch Gladbach-Frankenforst",
    "tax_id": null,
    "phone": null
  },
  "transaction": {
    "date": "2023-09-05",
    "time": "15:43",
    "currency": "EUR",
    "total_amount": 63.59,
    "payment_method": "girocard",
    "card_last_four": "7004",
    "receipt_number": "7620"
  },
  "items": [
    {
      "line_number": 1,
      "sku": "825667",
      "name": "Bio NL Kokosmilch",
      "quantity": 1.0,
      "unit": null,
      "unit_price": 0.99,
      "total_price": 0.99,
      "tax_category": "A",
      "discount": null
    },
    {
      "line_number": 2,
      "sku": "825667",
      "name": "Bio NL Kokosmilch",
      "quantity": 1.0,
      "unit": null,
      "unit_price": 0.99,
      "total_price": 0.99,
      "tax_category": "A",
      "discount": null
    },
    {
      "line_number": 3,
      "sku": "722061",
      "name": "Bio-Milch 3.8% 1L",
      "quantity": 

In [49]:
receipt.transaction

Transaction(date='2023-09-05', time='15:43', currency='EUR', total_amount=63.59, payment_method='girocard', card_last_four='7004', receipt_number='7620')

In [17]:
from_json(ergebnisse[0]['text'], allow_partial=False)

ValueError: expected value at line 1 column 1

In [15]:
ergebnisse[0]['text']

'```json\n{\n  "merchant": {\n    "name": "ALDI SÜD",\n    "address": "Frankenforster Str. 21-25, 51427 Bergisch Gladbach-Frankenforst",\n    "tax_id": null,\n    "phone": null\n  },\n  "transaction": {\n    "date": "2023-09-05",\n    "time": "15:43",\n    "currency": "EUR",\n    "total_amount": 63.59,\n    "payment_method": "girocard",\n    "card_last_four": "7004",\n    "receipt_number": "7620"\n  },\n  "items": [\n    {\n      "line_number": 1,\n      "sku": "825667",\n      "name": "Bio NL Kokosmilch",\n      "quantity": 1,\n      "unit": null,\n      "unit_price": 0.99,\n      "total_price": 0.99,\n      "tax_category": "A",\n      "discount": null\n    },\n    {\n      "line_number": 2,\n      "sku": "825667",\n      "name": "Bio NL Kokosmilch",\n      "quantity": 1,\n      "unit": null,\n      "unit_price": 0.99,\n      "total_price": 0.99,\n      "tax_category": "A",\n      "discount": null\n    },\n    {\n      "line_number": 3,\n      "sku": "722061",\n      "name": "Bio-Milc